# Figure 2 · Repeatability and representation checks

**问题：同一种细菌诱发的响应组合能否跨动物重现？** 主图比较同菌株与不同菌株的跨半样本相似度；支持图保留完整矩阵、可用拆分次数，以及未筛选模板与原始曲线的对照。

这里读取已有的 100 次整动物拆分结果，不重新分配动物或拟合模板。对每株或每个唯一菌株对，先对其有效拆分相似度求平均，再在图上出现一次。相似度使用完整八箱 0–40 s 重建，至少四类共同可用神经元；零范数的相似度未定义。

完成标准：主图计数与分布和原结果完全一致；支持检查明确区分“可重复性”与“筛选是否提高预测”。

In [ ]:
from pathlib import Path
import sys

# Run from the repository root or any directory below it (including poster/notebooks).
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "poster/src/poster_analysis").is_dir() and (p / "reports").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from this repository or its poster/notebooks directory")
sys.path.insert(0, str(ROOT / "poster/src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poster_analysis.paths import report_path, figure_dir, table_dir
from poster_analysis.style import apply_style, save_figure
from poster_analysis.constants import NEURONS, SEED

apply_style()
REPORT = report_path("responses")
TABLES = REPORT / "tables"
MAIN, SUPPORT, EXPORT_TABLES = figure_dir("main"), figure_dir("supporting"), table_dir()
np.random.seed(SEED)

from poster_analysis.repeatability import (
    load_repeatability_data, extract_distributions, histogram_summary,
    plot_repeatability, plot_similarity_matrix, aggregate_heldout_errors, COLORS,
)

BINS = np.arange(-1.0, 1.0001, 0.05)  # unchanged empirical histogram edges

## 1. 数据与来源

两个半样本按已有记录批次内的整只动物划分，所有 trial 随其动物进入同一半；两半独立筛选、拟合模板，均要求每个条件 × 神经元至少两只动物。以下源文件保存的是此前已完成的计算，不是当前 notebook 的新拟合。

In [ ]:
sources = pd.DataFrame([
    ("Per-pair mean over valid animal splits", "tables/split_filtered_cosine.csv"),
    ("Number of valid splits per pair", "tables/split_filtered_valid_splits.csv"),
    ("Common-support representation comparisons", "tables/split_control_pairs.csv"),
    ("Held-out raw-target errors per animal", "tables/loao_errors.csv"),
    ("Previously aggregated neuron-level errors", "tables/loao_cell_summary.csv"),
], columns=["Role", "Source relative to report"])
display(sources)
print("Source report:", REPORT.relative_to(ROOT))

data = load_repeatability_data(REPORT)
split, support = data["split"], data["support"]
values = split.to_numpy(float)
assert values.shape == (106, 106)
print("Saved similarity matrix:", values.shape)
print("Same-stimulus valid splits:", int(np.diag(support).min()), "to", int(np.diag(support).max()))

## 2. 显式提取同菌株与唯一异菌株对

对角线 $S_{ii}$ 表示同菌株跨半样本相似度；严格上三角 $S_{ij},i<j$ 表示唯一异菌株对。矩阵已对两个方向对称平均。每个有限条目权重相同；这里不按有效拆分次数加权，也不把 100 次拆分当作 100 个独立生物重复。

不同菌株的组合数是 $106\times105/2=5565$；其中没有定义相似度的对保留为缺失，不把它们设为零。

In [ ]:
same_all = np.diag(values)
upper = np.triu_indices(len(split), k=1)
different_all = values[upper]
same = same_all[np.isfinite(same_all)]
different = different_all[np.isfinite(different_all)]
n_undefined = int((~np.isfinite(different_all)).sum())
assert (len(same), len(different), n_undefined) == (106, 5396, 169)
extracted = extract_distributions(values)
np.testing.assert_array_equal(same, extracted["same"])
np.testing.assert_array_equal(different, extracted["different"])
summary = pd.DataFrame([
    {"Group": "Same stimulus", "n": len(same), "Mean": same.mean(), "Median": np.median(same)},
    {"Group": "Different stimuli", "n": len(different), "Mean": different.mean(), "Median": np.median(different)},
]).set_index("Group")
display(summary.round(5))
summary.to_csv(EXPORT_TABLES / "fig02_distribution_summary.csv")

## 3. 在 notebook 内构造密度分布，再导出紧凑主图

每组分开做单位面积归一化：$d_b=n_b/(N\,\Delta b)$。这使两组形状可以比较，不会让异菌株对因数量多而支配图面。完整范围 −1 到 1 保留负尾部；不平滑、不截取最好的样本。虚线表示均值。

In [ ]:
# Expose the empirical density calculation rather than hiding it in a plot call.
fig, ax = plt.subplots(figsize=(7.4, 3.7), layout="constrained")
histogram_rows = []
for group, scores, color in [("Different stimuli", different, COLORS["different"]),
                             ("Same stimulus", same, COLORS["same"])]:
    counts, edges = np.histogram(scores, bins=BINS)
    density = counts / (len(scores) * np.diff(edges))
    assert counts.sum() == len(scores)
    np.testing.assert_allclose(np.sum(density * np.diff(edges)), 1.0)
    np.testing.assert_allclose(density, histogram_summary(scores, BINS)["density"])
    ax.stairs(density, edges, fill=True, alpha=.27, color=color)
    ax.stairs(density, edges, color=color, label=group)
    ax.axvline(scores.mean(), color=color, ls="--", lw=1.3)
    histogram_rows.extend({"group": group, "left": edges[b], "right": edges[b+1],
                           "count": counts[b], "density": density[b]} for b in range(len(counts)))
ax.set(xlim=(-1, 1), ylim=(0, None), xlabel="Cross-half cosine similarity", ylabel="Density")
ax.legend(frameon=False)
plt.show()
pd.DataFrame(histogram_rows).to_csv(EXPORT_TABLES / "fig02_histogram_values.csv", index=False)

In [ ]:
# The reusable layout receives only the already audited distributions.
fig = plot_repeatability(same, different, bins=BINS)
save_figure(fig, MAIN / "fig02_repeatability")
plt.show()

同菌株分布总体偏向更高相似度，但两组存在重叠。该比较描述跨动物一致性，不等价于每株都可被准确分类。重叠菌株对及重复拆分并非独立观察，因此本图不使用将全部 pair 视为独立样本的显著性检验。

## 4. 支持图：完整矩阵和有效拆分支持

矩阵按 Figure 1 的固定顺序排列。灰色为未定义值；低于 −0.5 的相似度在矩阵上颜色饱和，但主图完整保留所有有限值。支持图显示有效拆分数，便于判断每个平均值由多少次拆分贡献。

In [ ]:
fig = plot_similarity_matrix(split, data["sample_order"])
save_figure(fig, SUPPORT / "fig02_full_similarity_matrix")
plt.show()

fig, ax = plt.subplots(figsize=(7.4, 3.3), layout="constrained")
count_bins = np.arange(-.5, 101.5, 5)
for counts, label, color in [(support.to_numpy()[upper], "Different stimuli", COLORS["different"]),
                             (np.diag(support), "Same stimulus", COLORS["same"])]:
    ax.hist(counts, bins=count_bins, density=True, histtype="step", lw=1.7, label=label, color=color)
ax.set(xlabel="Valid splits per stimulus or pair (of 100)", ylabel="Density", xlim=(-1, 101))
ax.legend(frameon=False)
save_figure(fig, SUPPORT / "fig02_split_support")
plt.show()

## 5. 支持探索：相同支持范围内比较表示方法

上面的主图使用筛选后重建的所有有效条目。若要比较表示方法，必须限定为相同细胞和相同有效拆分，避免支持范围不同影响比较。保存的 `split_control_pairs.csv` 已按这个共同支持规则生成。

下图只取该表的对角线，比较原始曲线、未筛选模板和筛选模板。它回答方法敏感性，不是新增选择“更好”阈值的步骤。

In [ ]:
controls = data["control_pairs"]
diagonal_controls = controls.loc[controls.diagonal.eq(True)].set_index("strain_a")
methods = {"raw": ("Raw curves", "#777777"),
           "unfiltered": ("Template, no gate", "#467BA5"),
           "filtered": ("Template, individual gate", "#B2182B")}
common_mask = np.isfinite(diagonal_controls[list(methods)]).all(axis=1)
matched = diagonal_controls.loc[common_mask, list(methods)]
display(matched.agg(["count", "mean", "median"]).T.round(5))

fig, ax = plt.subplots(figsize=(7.6, 3.7), layout="constrained")
for method, (label, color) in methods.items():
    counts, edges = np.histogram(matched[method], bins=BINS)
    density = counts / (len(matched) * np.diff(edges))
    ax.stairs(density, edges, color=color, lw=1.6, label=label)
ax.set(xlim=(-1, 1), ylim=(0, None), xlabel="Same-stimulus cross-half cosine", ylabel="Density")
ax.legend(frameon=False, fontsize=9)
save_figure(fig, SUPPORT / "fig02_matched_representation_controls")
plt.show()
matched.to_csv(EXPORT_TABLES / "fig02_matched_same_stimulus.csv")

## 6. 支持探索：预测被留出的动物

可重复性高不保证更好地预测未参与训练的动物。既有 leave-one-animal-out (LOAO) 分析在训练时去掉该动物及其全部 trial，重新筛选、拟合模板；验证目标是未置零的真实八箱均值。

以下显式按“同条件内动物等权 → 同株可用条件等权 → 每类神经元内菌株等权”聚合 MSE，最后开方得到 RMSE。不能先对每个观测开方再平均，也不能让记录更多的株获得更大的权重。

In [ ]:
errors = data["loao_errors"]
columns = ["mse_filtered", "mse_unfiltered", "mse_zero"]
scored = errors.loc[errors.scored.eq(True)]
condition_errors = scored.groupby(["sample_id", "block", "cell"])[columns].mean()
strain_errors = condition_errors.groupby(["sample_id", "cell"])[columns].mean()
cell_errors = strain_errors.groupby("cell")[columns].mean().reindex(NEURONS)
np.testing.assert_allclose(cell_errors, data["loao_cells"].loc[NEURONS, columns], rtol=1e-12, atol=1e-14)
np.testing.assert_allclose(cell_errors, aggregate_heldout_errors(errors).loc[NEURONS], rtol=1e-12, atol=1e-14)
rmse = np.sqrt(cell_errors)
rmse.columns = ["Individual gate", "No gate", "Zero response"]
display(rmse.round(4))
rmse.to_csv(EXPORT_TABLES / "fig02_heldout_neuron_rmse.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 5.1), layout="constrained")
y = np.arange(len(NEURONS))
x1, x2 = rmse["No gate"].to_numpy(), rmse["Individual gate"].to_numpy()
ax.hlines(y, np.minimum(x1, x2), np.maximum(x1, x2), color="#A9B4BF", lw=1.2)
ax.scatter(x1, y, s=38, color="#467BA5", label="No gate")
ax.scatter(x2, y, s=38, color="#B2182B", label="Individual gate")
ax.set(yticks=y, yticklabels=NEURONS, ylim=(len(y)-.5, -.5),
       xlabel=r"Held-out raw-target RMSE ($\Delta F/F_0$)")
ax.legend(frameon=False)
save_figure(fig, SUPPORT / "fig02_heldout_prediction")
plt.show()
print("Equal-neuron mean MSE:")
display(cell_errors.mean().to_frame("MSE"))
print("Neurons with smaller MSE after the gate:", int((cell_errors.mse_filtered < cell_errors.mse_unfiltered).sum()), "/", len(NEURONS))

筛选规则并不在全部神经元上改善留出预测，因此主图只支持“同菌株响应组合跨动物有一致性”的表述，不能把这一结果写成“筛选提升了预测”。筛选模板与未筛选模板的差别、未定义 pair 和支持次数都保留在本 notebook 的支持图中。

## 7. 导出与核对记录

主图是紧凑的分布比较；完整矩阵和方法检查用于展示细节。主图不以 5396 个 pair 的数量冒充独立动物数。

In [ ]:
checks = {
    "n_strains": len(split), "n_same": len(same), "n_different": len(different),
    "n_undefined_different": n_undefined,
    "same_mean": float(same.mean()), "different_mean": float(different.mean()),
    "histogram_bin_width": float(np.diff(BINS)[0]), "each_distribution_has_unit_area": True,
    "valid_split_same_min": int(np.diag(support).min()),
    "valid_split_same_max": int(np.diag(support).max()),
    "n_scored_heldout_observations": len(scored),
    "heldout_aggregation_matches_saved": True,
    "source_report": str(REPORT.relative_to(ROOT)), "refit": False,
}
(EXPORT_TABLES / "fig02_checks.json").write_text(json.dumps(checks, indent=2) + "\n")
sources.to_csv(EXPORT_TABLES / "fig02_sources.csv", index=False)
display(pd.Series(checks, name="Value").to_frame())